# Notebook 04 — KMeans + PCA Comparison
**Project:** Credit Card Customer Profiling & Segmentation  
**Goal:** Compare KMeans clustering across three approaches — raw scaled data, PCA (90% variance), and PCA (3 components) — using Silhouette, Inertia, ARI and NMI.

---
| Step | Action |
|------|--------|
| 1 | KMeans on raw scaled data |
| 2 | PCA — find components for 90% variance |
| 3 | KMeans on PCA (90% variance) |
| 4 | KMeans on PCA (3 components) |
| 5 | Full comparison: Silhouette, Inertia, ARI, NMI |
| 6 | Best approach recommendation |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, adjusted_rand_score,
                              normalized_mutual_info_score)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

from src.data_loader import load_raw_data
from src.preprocessing import preprocess
from src.feature_selection import get_selected_features
from src.config import RANDOM_STATE, N_CLUSTERS

palette = ['#2196F3','#4CAF50','#F44336','#FF9800']

df_scaled, df_unscaled = preprocess(load_raw_data(), save=False)
selected = get_selected_features(df_scaled)
df_sel = df_scaled[selected]
print(f'Feature-selected data: {df_sel.shape}')

## 1. KMeans on Raw Scaled Data (Baseline)

In [ ]:
km_raw     = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
raw_labels = km_raw.fit_predict(df_sel)
raw_sil    = silhouette_score(df_sel, raw_labels)

print(f'Raw Scaled | K={N_CLUSTERS} | Silhouette={raw_sil:.4f} | Inertia={km_raw.inertia_:,.0f}')
for k, n in zip(*np.unique(raw_labels, return_counts=True)):
    print(f'  Cluster {k}: {n:,} ({n/len(raw_labels)*100:.1f}%)')

## 2. PCA Variance Analysis

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(df_sel)
cumvar   = np.cumsum(pca_full.explained_variance_ratio_)
n_90     = int(np.searchsorted(cumvar, 0.90)) + 1

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(1, len(cumvar)+1), pca_full.explained_variance_ratio_*100,
       color='#2196F3', edgecolor='white', alpha=0.8, label='Individual')
ax.plot(range(1, len(cumvar)+1), cumvar*100, marker='o', color='#F44336',
        lw=2, label='Cumulative')
ax.axhline(90, color='gray', ls='--', lw=1.2, label='90% threshold')
ax.axvline(n_90, color='#FF9800', ls='--', lw=1.5, label=f'{n_90} components = 90%')
ax.set_title('PCA — Cumulative Variance Explained', fontsize=13, fontweight='bold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.legend()
plt.tight_layout()
plt.show()
print(f'{n_90} components explain {cumvar[n_90-1]*100:.1f}% of variance')

## 3. KMeans on PCA (90% Variance)

In [ ]:
pca_n90    = PCA(n_components=n_90, random_state=RANDOM_STATE)
data_n90   = pca_n90.fit_transform(df_sel)
km_n90     = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
labels_n90 = km_n90.fit_predict(data_n90)
sil_n90    = silhouette_score(data_n90, labels_n90)
ari_n90    = adjusted_rand_score(raw_labels, labels_n90)
nmi_n90    = normalized_mutual_info_score(raw_labels, labels_n90)

print(f'PCA ({n_90} comp) | Silhouette={sil_n90:.4f} | Inertia={km_n90.inertia_:,.0f} | ARI={ari_n90:.3f} | NMI={nmi_n90:.3f}')

## 4. KMeans on PCA (3 Components)

In [ ]:
pca_3    = PCA(n_components=3, random_state=RANDOM_STATE)
data_3   = pca_3.fit_transform(df_sel)
km_3     = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
labels_3 = km_3.fit_predict(data_3)
sil_3    = silhouette_score(data_3, labels_3)
ari_3    = adjusted_rand_score(raw_labels, labels_3)
nmi_3    = normalized_mutual_info_score(raw_labels, labels_3)

print(f'PCA (3 comp) | Silhouette={sil_3:.4f} | Inertia={km_3.inertia_:,.0f} | ARI={ari_3:.3f} | NMI={nmi_3:.3f}')
print(f'Variance explained by 3 components: {pca_3.explained_variance_ratio_.sum()*100:.1f}%')

## 5. Full Comparison Table

In [ ]:
comparison = pd.DataFrame([
    {'Approach': 'Raw Scaled Data',           'Components': df_sel.shape[1],
     'Variance %': 100.0, 'Silhouette': round(raw_sil,4),
     'Inertia': round(km_raw.inertia_,0), 'ARI': '--', 'NMI': '--'},
    {'Approach': f'PCA ({n_90} comp, 90% var)', 'Components': n_90,
     'Variance %': round(cumvar[n_90-1]*100,1), 'Silhouette': round(sil_n90,4),
     'Inertia': round(km_n90.inertia_,0), 'ARI': round(ari_n90,3), 'NMI': round(nmi_n90,3)},
    {'Approach': 'PCA (3 components)',         'Components': 3,
     'Variance %': round(pca_3.explained_variance_ratio_.sum()*100,1),
     'Silhouette': round(sil_3,4),
     'Inertia': round(km_3.inertia_,0), 'ARI': round(ari_3,3), 'NMI': round(nmi_3,3)},
])

comparison.style.highlight_max(subset=['Silhouette'], color='#C8E6C9') \
                .highlight_min(subset=['Inertia'], color='#C8E6C9')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
approaches_short = ['Raw', f'PCA\n({n_90} comp)', 'PCA\n(3 comp)']
colors = ['#2196F3', '#4CAF50', '#F44336']

for ax, metric, vals in zip(axes,
    ['Silhouette', 'Inertia', 'Variance %'],
    [[raw_sil, sil_n90, sil_3],
     [km_raw.inertia_, km_n90.inertia_, km_3.inertia_],
     [100.0, cumvar[n_90-1]*100, pca_3.explained_variance_ratio_.sum()*100]]):
    bars = ax.bar(approaches_short, vals, color=colors, edgecolor='white', alpha=0.88)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
                f'{v:.4f}' if metric == 'Silhouette' else f'{v:,.0f}',
                ha='center', fontsize=10, fontweight='bold')
    note = '(higher is better)' if metric == 'Silhouette' else ('(lower is better)' if metric == 'Inertia' else '')
    ax.set_title(f'{metric}\n{note}', fontweight='bold')

plt.suptitle('KMeans: Raw vs PCA Clustering Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. PCA 3D Cluster Visualisation

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10, 7))
ax  = fig.add_subplot(111, projection='3d')
for cluster in np.unique(labels_3):
    mask = labels_3 == cluster
    ax.scatter(data_3[mask, 0], data_3[mask, 1], data_3[mask, 2],
               label=f'Cluster {cluster}', alpha=0.4, s=12,
               color=palette[cluster % len(palette)])
ax.set_title(f'KMeans on PCA 3D (Silhouette={sil_3:.4f})', fontweight='bold')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

## Conclusion

| Approach | Silhouette | Inertia | ARI | NMI | Verdict |
|---|---|---|---|---|---|
| Raw Scaled | 0.1636 | 136,126 | — | — | Baseline |
| PCA (13 comp, 90%) | 0.1868 | 120,001 | 0.964 | 0.940 | Better separation, high alignment |
| **PCA (3 comp)** | **0.3253** | **39,622** | 0.711 | 0.697 | **Best cluster quality** |

**Key Insight:** PCA with 3 components achieves the **highest Silhouette Score (0.3253)** — nearly 2x better than raw data — despite only explaining 47% of variance. Removing noise from high-dimensional space produces much tighter, better-separated clusters.

**ARI/NMI Trade-off:** PCA (3 comp) has lower ARI/NMI vs PCA (13 comp), meaning it discovers *different* segments — which is expected and acceptable when the goal is natural grouping, not replicating the raw-data structure.

> **Next:** `05_Hierarchical_Clustering.ipynb` — compare with Ward linkage Hierarchical Clustering.